In [ ]:
import anndata as ad
import os
import sys

# read the pipeline modules - read data from 10x h5 files, 10x mtx files, and h5ad files
from load_data_10x_h5 import load_single_10x_h5, read_multiple_10x_h5_samples
from load_data_10x_mtx import load_single_10x_sample, read_multiple_10x_samples
from load_data_h5ad_data import read_single_h5ad_file, read_multiple_h5ad_files
from load_data_tabular import read_tabular_file, load_tabular_folder

# read the pipeline modules - preprocess the data
from qc import run_qc
from normalise import normalise
from run_celltypist import run_celltypist
from save_h5ad_file import save_h5ad_file









In [ ]:
# we gonna test it out and see if it works
from pathlib import Path

 # need to convert the string to Path
input_path = Path("/Users/z5155527/Desktop/ICI_Foundation_2026/ST_ICI_data/converted/gse179994_obj.h5ad")
gse179994 = read_single_h5ad_file(input_path)
gse179994

In [ ]:
# depending on the dataset, I need to change it to dataset_path/dataset_id
gse179994.obs.rename(columns={"sample_path": "dataset_path", "sample_id": "dataset_id"}, inplace=True)
gse179994.obs

In [ ]:
# QC
gse179994.layers["counts"] = gse179994.X.copy()
gse179994 = run_qc(gse179994)

# Normalise
gse179994 = normalise(gse179994)

# Run CellTypist
gse179994 = run_celltypist(gse179994)


In [ ]:
print("Ensure the raw data is not changed")
print(gse179994.X[:10, :10].toarray())
print("Raw data:")
print(gse179994.raw.X[:10, :10].toarray())


In [ ]:
print("Check CellTypist prediction")
print("unique predicted labels:")
gse179994.obs["predicted_labels"].unique()



In [ ]:
from align_gene_to_hg38 import align_gene_to_hg38


In [ ]:
gff3_path = "/Users/z5155527/Desktop/ICI_Foundation_2026/supplementary/Homo_sapiens.GRCh38.115.gff3"
gtf_file = "/Users/z5155527/Desktop/ICI_Foundation_2026/supplementary/gencode.v19.chr_patch_hapl_scaff.annotation.gtf"
gse179994_aligned =align_gene_to_hg38(gse179994, gff3_path, gtf_file)

In [ ]:
# we may need to manually convert the dataset_path to string
gse179994_aligned.obs["dataset_path"] = gse179994_aligned.obs["dataset_path"].astype(str)


In [ ]:
from save_h5ad_file import save_h5ad_file
save_h5ad_file(gse179994_aligned, "/Users/z5155527/Desktop/ICI_Foundation_2026/ST_ICI_data/done/gse179994_processed.h5ad")